

This script conducts a convex hull validation on current acoustic localizations. It applies a geometry-based quality filter to remove acoustic localizations that fall outside the spatial footprint of the ARUs that detected the sound event.


For each confirmed detection, the script retrieves the (x, y) coordinates of the specific ARUs that contributed to that localization (stored per-detection in the CSV). It then constructs the convex hull of those detecting ARUs — the tightest polygon enclosing the detection footprint and checks whether the localization point falls inside the hull or within 15 metres of its boundary.

The 15 m buffer was selected based on empirical analysis of residual RMS distributions across buffer zones, and is constrained to less than half the inter-ARU spacing (35 m).

Analysis of residual RMS across buffer categories showed that detections in the 10–15 m zone had a lower median RMS (0.191 m) than those strictly inside the hull (0.295 m), confirming these are geometrically marginal but acoustically well-constrained localizations. The 15 m threshold was therefore validated as the most appropriate cutoff.

In [ ]:
!pip install shapely
import pandas as pd
import numpy as np
import json
from scipy.spatial import Delaunay
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, Point as ShapelyPoint
import matplotlib.pyplot as plt

# ----- LOAD CSV ---------------

df = pd.read_csv('/Volumes/BUworkspace/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/weml_confirmed_locations_tessa.csv')
#the csv above includes the locations of the ARUs therefore, I do not need to upload two files

# ------ HULL FUNCTIONS ----------

def point_in_hull(point, hull):
    #check if point is inside teh convex hull of the given points
    if len(hull) < 3:
        return False
    delaunay = Delaunay(hull)
    return delaunay.find_simplex(point) >=0

def point_within_distance_of_hull(point, hull, distance_m=15):
    #check if a point is within distance_m of the convex hull of the given points
    if len(hull) < 3:
        #for degenerate cases, check distance to the hull points themselves
        distances = cdist([point], hull)[0]
        return distances.min() <= distance_m
    try:
        polygon = Polygon(hull)
        buffered = polygon.buffer(distance_m)
        return buffered.contains(ShapelyPoint(point))
    except Exception:
        #fallback: check if inside OR within distance of hull boundary
        delaunay = Delaunay(hull)
        if delaunay.find_simplex(point) >= 0:
            return True
        distances = cdist([point], hull)[0]
        return distances.min() <= distance_m

# --------- APPLY TO EACH ROW -------------------
def check_row(row):
    # The localization point (x, y only — 2D)
    point = [row['x'], row['y']]

    # The detecting ARU locations — parse from JSON, take only x and y
    receiver_locs = np.array(json.loads(row['receiver_locations']))[:, :2]

    inside_hull        = point_in_hull(point, receiver_locs)
    within_15m_of_hull = point_within_distance_of_hull(point, receiver_locs, distance_m=15)


    return pd.Series({
    'inside_hull':         inside_hull,
    'within_15m_of_hull':  within_15m_of_hull,
})

df[['inside_hull', 'within_15m_of_hull']] = df.apply(check_row, axis=1)

# --------- SUMMARY --------------------
print(f"Total detections:               {len(df)}")
print(f"Inside detecting ARU hull:      {df['inside_hull'].sum()}")
print(f"Within 15m of detecting ARU hull: {df['within_15m_of_hull'].sum()}")

print(f"test summary")
print("Buffer size vs detections retained:")
print(f"{'Buffer (m)':<15} {'Count':<10} {'% of total':<10}")
print("-" * 35)
for buffer in [5, 10, 15, 17, 20, 25, 35]:
    count = df.apply(
        lambda row: point_within_distance_of_hull(
            [row['x'], row['y']],
            np.array(json.loads(row['receiver_locations']))[:, :2],
            distance_m=buffer
        ), axis=1
    ).sum()
    print(f"{buffer:<15} {count:<10} {count/len(df)*100:.1f}%")

# --------- PLOTTING RMS VALUES --------
# Label each detection by which buffer category it falls into
def get_buffer_category(row):
    point = [row['x'], row['y']]
    receiver_locs = np.array(json.loads(row['receiver_locations']))[:, :2]

    if point_within_distance_of_hull(point, receiver_locs, distance_m=10):
        return 'inside_10m'
    elif point_within_distance_of_hull(point, receiver_locs, distance_m=15):
        return 'between_10_15m'
    else:
        return 'outside_15m'

df['buffer_category'] = df.apply(get_buffer_category, axis=1)

# Plot residual_rms distribution per category
fig, ax = plt.subplots(figsize=(9, 5))

colors = {'inside_10m': 'steelblue', 'between_10_15m': 'orange', 'outside_15m': 'red'}
labels = {'inside_10m': f'Inside 10m (n={sum(df["buffer_category"]=="inside_10m")})',
          'between_10_15m': f'10–15m zone (n={sum(df["buffer_category"]=="between_10_15m")})',
          'outside_15m': f'Outside 15m (n={sum(df["buffer_category"]=="outside_15m")})'}

for cat, color in colors.items():
    subset = df[df['buffer_category'] == cat]['residual_rms']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=labels[cat], edgecolor='none')

ax.set_xlabel('Residual RMS (m)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Residual RMS by hull buffer category', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# Also print summary stats per category
print("\nResidual RMS summary by category:")
print(df.groupby('buffer_category')['residual_rms'].describe().round(3))

# -------- FILTER -----------
df_filtered = df[df['within_15m_of_hull']].copy()
print(f"\nDetections after hull filter:   {len(df_filtered)}")

# --------- SAVE -----------
out_path = '/Volumes/BUworkspace/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/weml_confirmed_locations_hull15m_filtered_final.csv'
df_filtered.to_csv(out_path, index=False)
print(f"Saved filtered CSV to: {out_path}")